# Data Download

In [1]:
import sys
import urllib.request
import zipfile
from pathlib import Path

In [2]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
try:
    _notebook_dir = Path(__file__).resolve().parent
except NameError:
    _notebook_dir = Path().resolve()  # Jupyter: cwd is the notebook's directory

RAW_DIR = _notebook_dir.parent / "data" / "raw"
DEST_DIR = RAW_DIR / "movielens"
ZIP_PATH = RAW_DIR / "ml-100k.zip"
REQUIRED_FILES = ["u.data", "u.item"]

In [2]:
def _progress_hook(block_num: int, block_size: int, total_size: int) -> None:
    downloaded = block_num * block_size
    if total_size > 0:
        pct = min(downloaded * 100 / total_size, 100)
        print(f"\r  {pct:5.1f}%  ({downloaded // 1024} / {total_size // 1024} KB)", end="", flush=True)

In [6]:
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(MOVIELENS_URL, ZIP_PATH, reporthook=_progress_hook)

  100.0%  (4816 / 4808 KB)

(WindowsPath('Z:/Personal project/recommendation_system/data/raw/ml-100k.zip'),
 <http.client.HTTPMessage at 0x1bffdc66d50>)

In [7]:
DEST_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    # ml-100k.zip contains a top-level ml-100k/ folder; strip it
    for member in zf.infolist():
        # member.filename looks like "ml-100k/u.data"
        parts = Path(member.filename).parts
        if len(parts) < 2:
            continue
        relative = Path(*parts[1:])
        target = DEST_DIR / relative
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(zf.read(member.filename))

In [ ]:
list = [f for f in REQUIRED_FILES if not (DEST_DIR / f).exists()]
if list:
    raise FileNotFoundError(f"Missing files after extraction: {list}")


[]


In [5]:
for name in REQUIRED_FILES:
    size = (DEST_DIR / name).stat().st_size
    print(f"  {name}: {size // 1024} KB")

  u.data: 1932 KB
  u.item: 230 KB


# Data Preprocessing

In [3]:
raw_data_path = DEST_DIR / "u.data"
processed_data_path = _notebook_dir.parent / "data" / "processed"
matrix_file = processed_data_path / "movielens_matrix.npz"
user_map = processed_data_path / "user_mapping.json"
item_map = processed_data_path / "item_mapping.json"

MIN_rateing = 5

In [4]:
import json
import numpy as np
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

In [5]:
df = pd.read_csv(raw_data_path, sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
df.head(5)

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   item_id    100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


In [7]:
counts = df["user_id"].value_counts()
active_users = counts[counts >= MIN_rateing].index
filtered_df = df[df["user_id"].isin(active_users)]
print(f"Original ratings: {len(df)}, After filtering: {len(filtered_df)}")

Original ratings: 100000, After filtering: 100000


In [8]:
user_index = sorted(filtered_df["user_id"].unique())
item_index = sorted(filtered_df["item_id"].unique())
user_id_to_idx = {uid: idx for idx, uid in enumerate(user_index)}
item_id_to_idx = {iid: idx for idx, iid in enumerate(item_index)}

In [9]:
filtered_df["user_idx"] = filtered_df["user_id"].map(user_id_to_idx)
filtered_df["item_idx"] = filtered_df["item_id"].map(item_id_to_idx)

In [10]:
matrix = csr_matrix(
    (filtered_df["rating"].values, (filtered_df["user_idx"].values, filtered_df["item_idx"].values)),
    shape=(len(user_id_to_idx), len(item_id_to_idx)),
    dtype=np.float32,
)

In [14]:
matrix_file.parent.mkdir(parents=True, exist_ok=True)
user_map.parent.mkdir(parents=True, exist_ok=True)
item_map.parent.mkdir(parents=True, exist_ok=True)
save_npz(matrix_file, matrix)
print(f"matrix de taille : {matrix_file.stat().st_size / 1024}")
user_map.write_text(json.dumps({str(k) : v for k, v in user_id_to_idx.items()}, indent=2))
item_map.write_text(json.dumps({str(k) : v for k, v in item_id_to_idx.items()}, indent=2))


matrix de taille : 162.337890625


24697